# Analisi esplorativa dei dataset UniMiB e UniZgFall

Notebook **esclusivamente esplorativo e descrittivo**: tutte le analisi sono eseguite in memoria, nessun file viene scritto, modificato o ricampionato, nessun modello viene addestrato.

**Come leggere i risultati.** Ogni conclusione è marcata come *osservata* (calcolata dai dati in questo notebook) oppure come *ipotesi / dedotta* (con il motivo). Le informazioni non ricavabili sono dichiarate come *non disponibili*.

**Fonti dei dati.** Le registrazioni originali sono i CSV letti tramite i percorsi definiti in `config.py` (`DATASETS`). Le funzioni di `utils.py` per elenco file, label e soggetti sono riutilizzate. `utils.load_recording` scarta la 4ª colonna di UniZgFall, quindi qui i CSV sono riletti con `np.genfromtxt` (stesso parser) mantenendo **tutte** le colonne e gestendo gli errori per singolo file.

**Percorso:** configurazione e import; caricamento e controlli di qualità; distribuzione delle lunghezze; esplorazione interattiva; ispezione dei segnali e riepilogo.

**Prerequisito:** eseguire il notebook con `fall_detection_ml/` come directory di lavoro. Plotly e ipywidgets sono usati solo se già installati; in caso contrario viene stampato un avviso e nulla viene installato.

## Import e configurazione

Le dipendenze opzionali per i grafici e i controlli interattivi vengono rilevate senza installare pacchetti.

In [1]:
import sys
sys.dont_write_bytecode = True  # evita di scrivere file .pyc durante gli import

import hashlib
import importlib.util
import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import clear_output, display

import config as cfg
import utils

HAS_PLOTLY = importlib.util.find_spec('plotly') is not None
HAS_WIDGETS = importlib.util.find_spec('ipywidgets') is not None
if HAS_PLOTLY:
    import plotly.express as px
    import plotly.io as pio
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    pio.templates.default = 'plotly_white'  # stile unico per tutti i grafici
if HAS_WIDGETS:
    import ipywidgets as widgets

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 180)
pd.set_option('display.max_rows', 60)
pd.set_option('display.max_colwidth', 200)

missing = [n for n, ok in (('plotly', HAS_PLOTLY), ('ipywidgets', HAS_WIDGETS)) if not ok]
if missing:
    print('ATTENZIONE: dipendenze mancanti:', missing, '- i grafici/controlli corrispondenti non saranno disponibili (nessuna installazione eseguita).')
else:
    print('Plotly e ipywidgets disponibili.')
print('Dataset configurati in config.py:', list(cfg.DATASETS))

Plotly e ipywidgets disponibili.
Dataset configurati in config.py: ['UniZgFall', 'UniMiB']


## 1. Caricamento e inventario dei dataset

Il caricamento avviene **una sola volta**: tutte le registrazioni vengono lette in memoria (`SIG`) insieme a una tabella di metadati per registrazione (`REC`). Gli errori di lettura vengono registrati per singolo file senza interrompere l'esecuzione.

Le label sono associate ai file **per posizione** (dopo ordinamento naturale, come in `utils.build_dataset_files`): la cella di verifica sotto controlla come le label sono realmente rappresentate.

In [2]:
AXIS_COLS = 3  # come utils.load_recording: le prime 3 colonne sono gli assi x, y, z
AXIS_COLORS = ['#1f77b4', '#d62728', '#2ca02c']  # X, Y, Z


def rel(path):
    """Percorso relativo alla cartella del progetto (nessun percorso assoluto nei risultati)."""
    return os.path.relpath(path, cfg.PROJECT_DIR).replace(os.sep, '/')


def channel_names(n_cols):
    base = ['X', 'Y', 'Z']
    return [base[i] if i < 3 else f'col{i + 1}' for i in range(n_cols)]


def read_first_line(path):
    try:
        with open(path, 'r', errors='replace') as fh:
            return fh.readline().strip()
    except OSError:
        return None


def read_recording_full(path):
    """Legge un CSV originale mantenendo tutte le colonne. Ritorna (array 2D, errore)."""
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            a = np.genfromtxt(path, delimiter=',')
    except Exception as exc:
        return None, f'{type(exc).__name__}: {str(exc)[:120]}'
    a = np.asarray(a, dtype=float)
    if a.size == 0:
        return np.empty((0, 0)), None
    return np.atleast_2d(a), None


def load_dataset(name):
    info = cfg.DATASETS[name]
    res = {'available': False, 'notes': [], 'rec': None, 'sig': None, 'labels_raw': None}
    if not info['data_dir'].is_dir():
        res['notes'].append('cartella dati non trovata')
        return res
    files = utils.list_files(name)
    if not files:
        res['notes'].append('nessun file di registrazione trovato')
        return res
    labels = None
    if info['labels_file'].is_file():
        try:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                raw_labels = np.atleast_1d(np.loadtxt(info['labels_file']))
            if raw_labels.size == 0:
                res['notes'].append('file delle label vuoto')
            else:
                res['labels_raw'] = raw_labels
                lab = np.atleast_1d(utils.load_labels(name))
                if len(lab) == len(files):
                    labels = lab
                else:
                    res['notes'].append(f'{len(files)} file ma {len(lab)} label: associazione non possibile')
        except Exception as exc:
            res['notes'].append(f'errore di lettura delle label: {exc}')
    else:
        res['notes'].append('file delle label non trovato')
    names = info['class_names']
    rows, sigs = [], []
    for k, fname in enumerate(files):
        path = info['data_dir'] / fname
        a, err = read_recording_full(path)
        try:
            subject = utils.parse_subject(name, fname)
        except AssertionError:
            subject = np.nan
        lab = None if labels is None else int(labels[k])
        if lab is None:
            cname = 'N/D'
        elif 0 <= lab < len(names):
            cname = names[lab]
        else:
            cname = 'fuori mappa'
        ok = err is None and a.size > 0
        rows.append({
            'file': fname, 'label': lab, 'class_name': cname, 'subject': subject,
            'header': read_first_line(path),
            'n_rows': np.nan if err else (a.shape[0] if a.size else 0),
            'n_cols': np.nan if err else (a.shape[1] if a.size else 0),
            'n_nonfinite': np.nan if err else int((~np.isfinite(a)).sum()),
            'read_error': err,
        })
        sigs.append(a if ok else None)
    rec = pd.DataFrame(rows)
    rec['label'] = pd.to_numeric(rec['label']).astype('Int64')
    res.update(available=True, rec=rec, sig=sigs)
    return res


DATA = {name: load_dataset(name) for name in cfg.DATASETS}
REC = {n: d['rec'] for n, d in DATA.items() if d['available']}
SIG = {n: d['sig'] for n, d in DATA.items() if d['available']}


def V(name):
    """Registrazioni valide: lette senza errori e con almeno una riga."""
    r = REC[name]
    return r[r['read_error'].isna() & (r['n_rows'] > 0)]


for n, d in DATA.items():
    if d['available']:
        print(f'{n}: {len(REC[n])} file letti, {len(V(n))} registrazioni valide')
    else:
        print(f'{n}: NON disponibile ({d["notes"]})')

UniZgFall: 468 file letti, 468 registrazioni valide
UniMiB: 1980 file letti, 1980 registrazioni valide


In [3]:
for name, d in DATA.items():
    info = cfg.DATASETS[name]
    print('=' * 88)
    print(name)
    print('  cartella dati    :', rel(info['data_dir']))
    print('  file delle label :', rel(info['labels_file']))
    if not d['available']:
        print('  DATASET NON DISPONIBILE:', '; '.join(d['notes']))
        continue
    rec = d['rec']
    ext = rec['file'].map(lambda f: Path(f).suffix or '(nessuna)').value_counts().to_dict()
    print('  registrazioni (file):', len(rec), '| estensioni:', ext)
    hdr = rec['header'].fillna('')
    is_hdr = hdr.str.startswith('#')
    print(f'  file con prima riga commentata (#): {int(is_hdr.sum())} su {len(rec)}', end='')
    print(' | testi distinti:', hdr[is_hdr].unique().tolist() if is_hdr.any() else '(nessuna intestazione)')
    print('  colonne per registrazione (n_colonne -> n_file):', rec['n_cols'].value_counts(dropna=False).to_dict())
    modes = rec['n_cols'].mode()
    if modes.empty:
        print('  canali (colonne) piu comuni: non disponibili (nessun file interpretabile)')
    else:
        modal = int(modes.iloc[0])
        print('  canali (colonne) piu comuni:', modal, '-> nomi usati nel notebook:', channel_names(modal))
    if name == 'UniZgFall':
        print('  nota: nessuna intestazione nel file; X, Y, Z = prime 3 colonne e 4a colonna = tempo sono ASSUNZIONI del progetto (README), non dichiarate nei file')
    subj = rec['subject'].dropna()
    if subj.empty:
        print('  identificativo soggetto: NON disponibile (nessun ID nei nomi dei file)')
    else:
        per = subj.value_counts()
        print(f'  soggetti distinti: {subj.nunique()} | registrazioni per soggetto: min {per.min()}, max {per.max()} (ID ricavato dal nome file con config.DATASETS[...][subject_regex])')
    names = info['class_names']
    cnt = rec['label'].value_counts()
    tab = pd.DataFrame({'label': range(len(names)), 'classe': names})
    tab['n_registrazioni'] = tab['label'].map(cnt).fillna(0).astype(int)
    tab['%'] = (100 * tab['n_registrazioni'] / len(rec)).round(1)
    print(f'  classi nella mappatura di config.py: {len(names)} | classi con almeno una registrazione: {int((tab["n_registrazioni"] > 0).sum())}')
    display(tab.set_index('label'))
    n_err = int(rec['read_error'].notna().sum())
    if n_err:
        print(f'  ERRORI DI LETTURA: {n_err} file')
        display(rec.loc[rec['read_error'].notna(), ['file', 'read_error']].head(10))
    for note in d['notes']:
        print('  NOTA:', note)

UniZgFall
  cartella dati    : ../ml_accelerometers/UniZgFall
  file delle label : ../ml_accelerometers/y_UniZgFall.csv
  registrazioni (file): 468 | estensioni: {'.csv': 468}
  file con prima riga commentata (#): 0 su 468 | testi distinti: (nessuna intestazione)
  colonne per registrazione (n_colonne -> n_file): {4: 468}
  canali (colonne) piu comuni: 4 -> nomi usati nel notebook: ['X', 'Y', 'Z', 'col4']
  nota: nessuna intestazione nel file; X, Y, Z = prime 3 colonne e 4a colonna = tempo sono ASSUNZIONI del progetto (README), non dichiarate nei file
  identificativo soggetto: NON disponibile (nessun ID nei nomi dei file)
  classi nella mappatura di config.py: 10 | classi con almeno una registrazione: 10


,classe,n_registrazioni,%
label,,,
0,MW,68,14.5
1,MR,68,14.5
2,MJ,68,14.5
3,WD,13,2.8
4,WU,13,2.8
5,FF,34,7.3
6,FS,34,7.3
7,FB,34,7.3
8,LD,34,7.3


UniMiB
  cartella dati    : ../ml_accelerometers/UniMiB_raw
  file delle label : ../ml_accelerometers/y_UniMiB_SHAR.csv
  registrazioni (file): 1980 | estensioni: {'.csv': 1980}
  file con prima riga commentata (#): 1980 su 1980 | testi distinti: ['# x,y,z']
  colonne per registrazione (n_colonne -> n_file): {3: 1980}
  canali (colonne) piu comuni: 3 -> nomi usati nel notebook: ['X', 'Y', 'Z']
  soggetti distinti: 30 | registrazioni per soggetto: min 66, max 66 (ID ricavato dal nome file con config.DATASETS[...][subject_regex])
  classi nella mappatura di config.py: 17 | classi con almeno una registrazione: 17


,classe,n_registrazioni,%
label,,,
0,Walking,60,3.0
1,Running,60,3.0
2,GoingUpS,60,3.0
3,GoingDownS,60,3.0
4,Jumping,60,3.0
5,SittingDown,60,3.0
6,StandingUpFS,60,3.0
7,LyingDownFS,60,3.0
8,StandingUpFL,60,3.0


In [4]:
for name, d in DATA.items():
    if not d['available']:
        continue
    info = cfg.DATASETS[name]
    rec = d['rec']
    raw = d['labels_raw']
    names = info['class_names']
    print('=' * 88)
    print(name, '- verifica della rappresentazione delle label')
    if raw is None:
        print('  file delle label non leggibile: verifica non possibile')
        continue
    print('  file:', rel(info['labels_file']), '| prima riga:', read_first_line(info['labels_file']))
    print('  valori letti:', len(raw), '| tutti interi (anche se scritti in notazione scientifica float):', bool(np.all(np.mod(raw, 1) == 0)))
    print('  minimo/massimo:', raw.min(), raw.max(), '| classi nella mappatura:', len(names))
    obs = set(int(x) for x in raw)
    print('  label osservate non presenti nella mappatura:', sorted(obs - set(range(len(names)))) or 'nessuna')
    print('  classi della mappatura mai osservate:', [names[i] for i in range(len(names)) if i not in obs] or 'nessuna')
    print('  il file label contiene solo numeri: l associazione ai file e POSIZIONALE (ordinamento naturale in utils.list_files)')
    if name == 'UniZgFall':
        suf = rec['file'].map(lambda f: Path(f).stem.rsplit('_', 1)[-1])
        mism = int((suf != rec['class_name']).sum())
        print(f'  controllo indipendente: suffisso del nome file vs nome classe di config.py -> {mism} discordanze su {len(rec)}')
        print('  (il significato delle sigle MW, MR, ... non e documentato nel repository: la corrispondenza sigla -> attivita non e verificabile)')
    if name == 'UniMiB':
        idx = rec['file'].map(lambda f: int(Path(f).stem.rsplit('_', 1)[-1]))
        per_idx = rec.assign(file_idx=idx).groupby('file_idx')['label'].nunique()
        print(f'  indice di file (file_N) -> n. label distinte: massimo {int(per_idx.max())}; ogni N ha una sola label: {bool((per_idx == 1).all())}')
        print('  il nome file non contiene il nome della classe: la mappatura posizionale label -> file NON e verificabile dai soli nomi')

UniZgFall - verifica della rappresentazione delle label
  file: ../ml_accelerometers/y_UniZgFall.csv | prima riga: # True_label
  valori letti: 468 | tutti interi (anche se scritti in notazione scientifica float): True
  minimo/massimo: 0.0 9.0 | classi nella mappatura: 10
  label osservate non presenti nella mappatura: nessuna
  classi della mappatura mai osservate: nessuna
  il file label contiene solo numeri: l associazione ai file e POSIZIONALE (ordinamento naturale in utils.list_files)
  controllo indipendente: suffisso del nome file vs nome classe di config.py -> 0 discordanze su 468
  (il significato delle sigle MW, MR, ... non e documentato nel repository: la corrispondenza sigla -> attivita non e verificabile)
UniMiB - verifica della rappresentazione delle label
  file: ../ml_accelerometers/y_UniMiB_SHAR.csv | prima riga: # True_label
  valori letti: 1980 | tutti interi (anche se scritti in notazione scientifica float): True
  minimo/massimo: 0.0 16.0 | classi nella mappatura:

## 2. Distribuzione delle lunghezze

La *lunghezza* di una registrazione è il numero di righe (campioni) del CSV originale. Sono considerate **valide** le registrazioni lette senza errori e con almeno una riga.

Per «registrazioni con lunghezza identica» sono riportate due misure: quante registrazioni hanno la lunghezza *più frequente* e quante condividono la propria lunghezza con almeno un'altra registrazione.

In [5]:
def length_stats(s):
    s = s.astype(int)
    vc = s.value_counts()
    q = s.quantile([0.25, 0.75, 0.90, 0.95, 0.99])
    return {
        'n_valide': len(s), 'min': int(s.min()), 'max': int(s.max()),
        'media': round(float(s.mean()), 1), 'mediana': float(s.median()),
        'std': round(float(s.std(ddof=1)), 1) if len(s) > 1 else np.nan,
        'cv': round(float(s.std(ddof=1) / s.mean()), 3) if len(s) > 1 else np.nan,
        'q25': float(q.loc[0.25]), 'q75': float(q.loc[0.75]), 'q90': float(q.loc[0.9]),
        'q95': float(q.loc[0.95]), 'q99': float(q.loc[0.99]),
        'lunghezza_piu_frequente': int(vc.index[0]),
        'n_con_lunghezza_piu_frequente': int(vc.iloc[0]),
        '%_con_lunghezza_piu_frequente': round(100 * float(vc.iloc[0]) / len(s), 1),
        'n_in_lunghezze_condivise': int(vc[vc > 1].sum()),
        '%_in_lunghezze_condivise': round(100 * float(vc[vc > 1].sum()) / len(s), 1),
        'n_lunghezze_distinte': int(len(vc)),
    }


LEN_ALL = pd.DataFrame.from_dict({n: length_stats(V(n)['n_rows']) for n in REC if len(V(n))}, orient='index')
print('Lunghezze per dataset (numero di righe/campioni per registrazione)')
display(LEN_ALL.T)

CLASS_STATS = {}
for name in REC:
    v = V(name)
    rows = {}
    for (lab, cname), g in v.groupby(['label', 'class_name']):
        rows[(int(lab), cname)] = length_stats(g['n_rows'])
    if not rows:
        print(f'{name}: statistiche per classe non disponibili (label assenti)')
        continue
    t = pd.DataFrame.from_dict(rows, orient='index')
    t.index = pd.MultiIndex.from_tuples(t.index, names=['label', 'classe'])
    t['uniforme'] = t['n_lunghezze_distinte'] == 1
    CLASS_STATS[name] = t
    print(f'\n{name}: lunghezze per classe')
    display(t)

anom = {}
for name, rec in REC.items():
    modes = rec['n_cols'].mode()
    modal = modes.iloc[0] if not modes.empty else np.nan
    anom[name] = {
        'file totali': len(rec),
        'errore di lettura / malformati': int(rec['read_error'].notna().sum()),
        'vuote (0 righe)': int((rec['n_rows'] == 0).sum()),
        'con valori non finiti (NaN/inf)': int((rec['n_nonfinite'] > 0).sum()),
        'con n. colonne diverso dal piu comune': int((rec['n_cols'] != modal).sum()) if pd.notna(modal) else 0,
    }
print('\nRegistrazioni vuote, malformate o non interpretabili')
display(pd.DataFrame(anom).T)

Lunghezze per dataset (numero di righe/campioni per registrazione)


,UniZgFall,UniMiB
n_valide,468.000,1980.000
min,388.000,85.000
max,5146.000,1863.000
media,1697.600,374.700
mediana,1525.000,206.000
std,803.000,380.400
cv,0.473,1.015
q25,1156.500,151.750
q75,2019.000,446.500
q90,2549.900,870.100



UniZgFall: lunghezze per classe


,,n_valide,min,max,media,mediana,std,cv,q25,q75,q90,q95,q99,lunghezza_piu_frequente,n_con_lunghezza_piu_frequente,%_con_lunghezza_piu_frequente,n_in_lunghezze_condivise,%_in_lunghezze_condivise,n_lunghezze_distinte,uniforme
label,classe,,,,,,,,,,,,,,,,,,,
0,MW,68,1399,3475,2302.6,2351.0,374.2,0.162,2010.75,2534.75,2732.5,2830.05,3199.63,2736,1,1.5,0,0.0,68,False
1,MR,68,755,2549,1739.5,1753.0,362.6,0.208,1552.00,1917.50,2247.3,2347.80,2494.73,2468,2,2.9,2,2.9,67,False
2,MJ,68,593,1891,1154.6,1107.0,283.1,0.245,978.50,1287.75,1638.8,1650.60,1803.23,987,2,2.9,8,11.8,64,False
3,WD,13,3157,5020,4029.0,3970.0,497.3,0.123,3763.00,4332.00,4459.0,4696.60,4955.32,4332,1,7.7,0,0.0,13,False
4,WU,13,3335,5146,4173.7,4062.0,584.5,0.140,3785.00,4661.00,4887.0,5006.80,5118.16,5146,1,7.7,0,0.0,13,False
5,FF,34,840,2707,1457.9,1398.5,428.9,0.294,1160.00,1691.00,1926.9,2199.75,2630.77,1160,2,5.9,2,5.9,33,False
6,FS,34,956,2542,1711.0,1647.0,396.9,0.232,1474.50,2019.75,2131.8,2301.95,2500.75,1618,2,5.9,2,5.9,33,False
7,FB,34,1176,3130,1978.1,1963.0,468.6,0.237,1662.50,2314.00,2447.6,2713.25,3085.78,1767,1,2.9,0,0.0,34,False
8,LD,34,824,2385,1298.3,1260.5,303.0,0.233,1102.00,1488.75,1587.4,1619.90,2135.52,1201,1,2.9,0,0.0,34,False



UniMiB: lunghezze per classe


,,n_valide,min,max,media,mediana,std,cv,q25,q75,q90,q95,q99,lunghezza_piu_frequente,n_con_lunghezza_piu_frequente,%_con_lunghezza_piu_frequente,n_in_lunghezze_condivise,%_in_lunghezze_condivise,n_lunghezze_distinte,uniforme
label,classe,,,,,,,,,,,,,,,,,,,
0,Walking,60,1470,1863,1599.3,1582.0,89.2,0.056,1533.50,1650.50,1712.2,1763.00,1843.53,1516,3,5.0,11,18.3,54,False
1,Running,60,1224,1731,1498.4,1505.0,68.7,0.046,1473.00,1529.25,1563.2,1586.10,1660.20,1500,2,3.3,8,13.3,56,False
2,GoingUpS,60,336,854,592.0,582.5,101.8,0.172,526.75,640.25,732.4,796.25,846.33,628,2,3.3,2,3.3,59,False
3,GoingDownS,60,412,830,601.0,594.5,93.4,0.155,526.00,644.50,736.4,773.45,824.10,592,2,3.3,10,16.7,55,False
4,Jumping,60,85,1159,537.0,493.5,183.3,0.341,420.00,617.25,739.0,963.70,1075.22,420,2,3.3,6,10.0,57,False
5,SittingDown,60,382,1133,653.4,591.0,201.0,0.308,480.75,836.25,934.3,970.50,1066.33,505,2,3.3,6,10.0,57,False
6,StandingUpFS,60,374,1215,655.5,564.0,208.3,0.318,486.25,828.25,928.4,970.50,1158.95,883,2,3.3,8,13.3,56,False
7,LyingDownFS,60,382,1678,862.3,681.5,346.7,0.402,572.25,1163.00,1301.1,1397.35,1669.74,1293,2,3.3,8,13.3,56,False
8,StandingUpFL,60,425,1746,846.7,675.0,365.2,0.431,528.75,1171.25,1377.4,1433.45,1662.22,511,2,3.3,6,10.0,57,False



Registrazioni vuote, malformate o non interpretabili


,file totali,errore di lettura / malformati,vuote (0 righe),con valori non finiti (NaN/inf),con n. colonne diverso dal piu comune
UniZgFall,468,0,0,0,0
UniMiB,1980,0,0,0,0


In [6]:
if not HAS_PLOTLY:
    print('Plotly non disponibile: grafici omessi.')
else:
    for name in REC:
        v = V(name)
        if v.empty:
            continue
        fig = px.histogram(v, x='n_rows', nbins=60,
                           title=f'{name}: distribuzione delle lunghezze originali (n. campioni per registrazione)',
                           labels={'n_rows': 'lunghezza (n. campioni)'})
        fig.update_layout(yaxis_title='n. registrazioni', bargap=0.05)
        fig.show()


### Confronto con i file derivati in `data/` (sola lettura)

Il progetto genera `data/raw_*_X.npy` (zero-padding) e `data/raw_*_meta.csv` (con `n_samples`). La cella legge questi file senza modificarli e verifica, registrazione per registrazione: (a) se `n_samples` coincide con la lunghezza dei CSV originali; (b) se la coda oltre `n_samples` è interamente nulla; (c) se la parte iniziale coincide esattamente con i dati originali.

In [7]:
def check_derived(name):
    meta_p = cfg.DATA_DIR / f'raw_{name}_meta.csv'
    x_p = cfg.DATA_DIR / f'raw_{name}_X.npy'
    out = {'stato': 'ok'}
    if not (meta_p.is_file() and x_p.is_file()):
        out['stato'] = 'file derivati (data/raw_*) non trovati: confronto non eseguito'
        return out
    try:
        meta = pd.read_csv(meta_p)
        X = np.load(x_p, mmap_mode='r')
    except Exception as exc:
        out['stato'] = f'lettura fallita: {exc}'
        return out
    pos = {f: i for i, f in enumerate(REC[name]['file'])}
    checked = eq_len = tail_nz = data_eq = 0
    for r, (fid, n) in enumerate(zip(meta['record_id'], meta['n_samples'])):
        i = pos.get(fid)
        if i is None or SIG[name][i] is None or r >= X.shape[0]:
            continue
        a = SIG[name][i][:, :AXIS_COLS]
        n = int(n)
        row = np.asarray(X[r])
        checked += 1
        eq_len += int(len(a) == n)
        tail_nz += int(np.any(row[n * AXIS_COLS:] != 0))
        data_eq += int(len(a) == n and np.array_equal(row[:n * AXIS_COLS], a.ravel()))
    L = X.shape[1] // AXIS_COLS
    tot = float(meta['n_samples'].sum())
    out.update({
        'forma array': tuple(X.shape), 'lunghezza_con_padding': L,
        'righe in meta': len(meta), 'registrazioni confrontate': checked,
        'n_samples == lunghezza CSV originale': eq_len,
        'con coda oltre n_samples NON nulla': tail_nz,
        'parte iniziale identica ai CSV': data_eq,
        'quota di campioni di padding (%)': round(100 * (1 - tot / (len(meta) * L)), 1),
    })
    return out


DERIVED = {name: check_derived(name) for name in REC}
display(pd.DataFrame(DERIVED).T)

,stato,forma array,lunghezza_con_padding,righe in meta,registrazioni confrontate,n_samples == lunghezza CSV originale,con coda oltre n_samples NON nulla,parte iniziale identica ai CSV,quota di campioni di padding (%)
UniZgFall,ok,"(468, 15450)",5150,468,468,468,0,468,67.0
UniMiB,ok,"(1980, 9624)",3208,1980,1980,1980,0,1980,88.3


In [8]:
try:
    from scipy import stats as sps
except ImportError:
    sps = None


def eta_squared(v):
    grp = v.groupby('label')['n_rows']
    gm = v['n_rows'].mean()
    ss_b = (grp.count() * (grp.mean() - gm) ** 2).sum()
    ss_t = ((v['n_rows'] - gm) ** 2).sum()
    return float(ss_b / ss_t) if ss_t > 0 else float('nan')


ETA2, KW = {}, {}
for name in REC:
    v = V(name)
    print('=' * 88)
    print(name)
    if v.empty:
        print('  nessuna registrazione valida')
        continue
    s = LEN_ALL.loc[name]
    nd = int(s['n_lunghezze_distinte'])
    # Domanda 1
    if nd == 1:
        print(f'  1) Stessa lunghezza effettiva per tutte le registrazioni? SI (osservato): {int(s["min"])} campioni.')
    else:
        print(f'  1) Stessa lunghezza effettiva per tutte le registrazioni? NO (osservato): {nd} lunghezze distinte su {int(s["n_valide"])} registrazioni; la piu frequente ({int(s["lunghezza_piu_frequente"])}) copre il {s["%_con_lunghezza_piu_frequente"]}%.')
    # Domanda 2
    cs = CLASS_STATS.get(name)
    if cs is None:
        print('  2) Uniformita per classe: non valutabile (label non disponibili).')
    else:
        n_uni = int(cs['uniforme'].sum())
        if n_uni == len(cs):
            print(f'  2) Lunghezze uniformi in ogni classe? SI (osservato): tutte le {len(cs)} classi hanno una sola lunghezza.')
        else:
            print(f'  2) Lunghezze uniformi in ogni classe? NO (osservato): solo {n_uni} classi su {len(cs)} hanno lunghezza unica; n. lunghezze distinte per classe da {int(cs["n_lunghezze_distinte"].min())} a {int(cs["n_lunghezze_distinte"].max())}; CV intra-classe da {cs["cv"].min():.2f} a {cs["cv"].max():.2f}.')
    # Domanda 3
    print(f'  3) Lunghezze (osservate): minima {int(s["min"])}, massima {int(s["max"])}, mediana {s["mediana"]:.0f}, media {s["media"]:.1f}; intervallo interquartile {s["q25"]:.0f}-{s["q75"]:.0f}; q99 {s["q99"]:.0f}.')
    # Domanda 4
    if cs is not None and len(cs) > 1:
        med = cs['mediana']
        ETA2[name] = eta_squared(v)
        kw_txt = ''
        groups = [g['n_rows'].values for _, g in v.groupby('label') if len(g) > 1]
        if sps is not None and len(groups) > 1:
            try:
                h, p = sps.kruskal(*groups)
                KW[name] = (float(h), float(p))
                kw_txt = f'; Kruskal-Wallis H={h:.1f}, p={p:.2g}'
            except ValueError:
                kw_txt = '; Kruskal-Wallis non calcolabile'
        print(f'  4) Differenze tra classi (osservato): mediane da {med.min():.0f} ({med.idxmin()[1]}) a {med.max():.0f} ({med.idxmax()[1]}), rapporto {med.max() / med.min():.1f}x; frazione di varianza della lunghezza spiegata dalla classe (eta^2) = {ETA2[name]:.2f}{kw_txt}.')
        print('     Interpretazione: valori elevati di eta^2 indicano che la lunghezza e un indizio della classe (rilevante per modelli sui segnali grezzi con padding); e una lettura descrittiva, non un test di causalita.')
    else:
        print('  4) Differenze tra classi: non valutabili.')
    # Domanda 5
    dv = DERIVED.get(name, {})
    print('  5) Lunghezza originale vs padding:')
    print(f'     - CSV originali (osservato): lunghezze variabili ({nd} valori distinti) e nessuna colonna/riga che indichi padding: sono registrazioni di lunghezza nativa.')
    if dv.get('stato') == 'ok':
        print(f'     - array derivato data/raw_{name}_X.npy (osservato): forma {dv["forma array"]}, padding fino a {dv["lunghezza_con_padding"]} campioni (circa {dv["quota di campioni di padding (%)"]}% dei campioni sono padding).')
        print(f'       n_samples di raw_{name}_meta.csv coincide con la lunghezza originale in {dv["n_samples == lunghezza CSV originale"]}/{dv["registrazioni confrontate"]} registrazioni; coda non nulla in {dv["con coda oltre n_samples NON nulla"]}; parte iniziale identica in {dv["parte iniziale identica ai CSV"]}.')
        print('       => la lunghezza originale e distinguibile dal padding SOLO tramite n_samples (meta): dal solo array non e deducibile con certezza (vedi Sezione 4).')
    else:
        print('     - confronto con i file derivati non eseguito:', dv.get('stato'))

UniZgFall
  1) Stessa lunghezza effettiva per tutte le registrazioni? NO (osservato): 427 lunghezze distinte su 468 registrazioni; la piu frequente (1159) copre il 0.6%.
  2) Lunghezze uniformi in ogni classe? NO (osservato): solo 0 classi su 10 hanno lunghezza unica; n. lunghezze distinte per classe da 13 a 97; CV intra-classe da 0.12 a 0.29.
  3) Lunghezze (osservate): minima 388, massima 5146, mediana 1525, media 1697.6; intervallo interquartile 1156-2019; q99 4540.
  4) Differenze tra classi (osservato): mediane da 1107 (MJ) a 4062 (WU), rapporto 3.7x; frazione di varianza della lunghezza spiegata dalla classe (eta^2) = 0.79; Kruskal-Wallis H=302.2, p=9e-60.
     Interpretazione: valori elevati di eta^2 indicano che la lunghezza e un indizio della classe (rilevante per modelli sui segnali grezzi con padding); e una lettura descrittiva, non un test di causalita.
  5) Lunghezza originale vs padding:
     - CSV originali (osservato): lunghezze variabili (427 valori distinti) e nessuna

## 3. Esplorazione interattiva: registrazioni e lunghezze

Seleziona una delle due viste. La prima aggiorna il segnale della registrazione scelta; la seconda confronta le lunghezze per classe. Ogni grafico viene aggiornato nella stessa area di output.

Due viste selezionabili con il pulsante **Vista** e indipendenti l'una dall'altra, entrambe legate al **dataset** scelto:

- **Registrazione**: scegliere classe e file; il grafico mostra X, Y, Z (prime 3 colonne, come `utils.load_recording`) e la magnitudo √(X²+Y²+Z²) in funzione del campione. `Max campioni` > 0 limita la visualizzazione ai primi N campioni (il titolo lo dichiara); 0 = serie completa. Le colonne extra (UniZgFall: 4ª colonna) non sono tracciate come segnale ma possono fungere da asse x.
- **Lunghezze per classe**: un box plot per classe (ordine di `config.py`) di `n_rows`, il numero di campioni della registrazione, con i singoli punti sovrapposti; il triangolo/linea tratteggiata indica la media.

Cambiando dataset o classe, le opzioni successive vengono ricostruite e il grafico viene disegnato **una sola volta** nell'area di output. Unità di misura e frequenza di campionamento non sono documentate (vedi Sezione 5).


In [15]:
MAG_COLOR = '#444444'
X_INDEX = 'indice del campione'
X_EXTRA = 'colonna extra del file (significato non documentato)'
VIEW_RECORDING, VIEW_LENGTHS = 'recording', 'lengths'

# Gli indici di REC sono le posizioni corrispondenti agli array in SIG.
for _name in REC:
    assert REC[_name].index.tolist() == list(range(len(SIG[_name]))), f'{_name}: indice di REC non allineato a SIG'


def magnitude(signal):
    """Modulo del vettore (X, Y, Z) per ogni campione."""
    return np.linalg.norm(signal[:, :AXIS_COLS], axis=1)


def class_order(name):
    """Classi con dati nell'ordine di config.py, poi eventuali classi extra."""
    present = list(V(name)['class_name'].dropna().unique())
    configured = [c for c in cfg.DATASETS[name]['class_names'] if c in present]
    return configured + [c for c in present if c not in configured]


def signal_figure(name, position, max_samples=0, x_mode=X_INDEX):
    """Crea il grafico temporale di una registrazione valida."""
    if position is None or not 0 <= int(position) < len(SIG[name]):
        return None
    signal = SIG[name][int(position)]
    if signal is None or signal.ndim != 2 or signal.shape[0] == 0 or signal.shape[1] == 0:
        return None

    row = REC[name].iloc[int(position)]
    n_total = len(signal)
    n_shown = min(int(max_samples), n_total) if 0 < int(max_samples) < n_total else n_total
    channels = channel_names(signal.shape[1])
    sample_index = np.arange(n_shown)
    x_values, x_label = sample_index, X_INDEX
    if x_mode == X_EXTRA and signal.shape[1] > AXIS_COLS:
        x_values = signal[:n_shown, AXIS_COLS]
        x_label = f'{channels[AXIS_COLS]} del file (unità e significato non documentati)'

    fig = go.Figure()
    for axis in range(min(AXIS_COLS, signal.shape[1])):
        fig.add_trace(go.Scatter(
            x=x_values, y=signal[:n_shown, axis], customdata=sample_index,
            mode='lines', name=channels[axis],
            line=dict(color=AXIS_COLORS[axis], width=1.2),
            hovertemplate='campione=%{customdata}<br>' + channels[axis] + '=%{y:.4f}<extra></extra>'))
    if signal.shape[1] >= AXIS_COLS:
        fig.add_trace(go.Scatter(
            x=x_values, y=magnitude(signal[:n_shown]), customdata=sample_index,
            mode='lines', name='Magnitudo',
            line=dict(color=MAG_COLOR, width=1.5, dash='dot'),
            hovertemplate='campione=%{customdata}<br>Magnitudo=%{y:.4f}<extra></extra>'))

    sample_note = f'{n_shown} di {n_total} campioni'
    if n_shown < n_total:
        sample_note += ' (visualizzati i primi)'
    fig.update_layout(
        title=(f'{name} | classe: {row["class_name"]} | file: {row["file"]}'
               f'<br><sup>{sample_note} | unità e frequenza di campionamento non documentate</sup>'),
        xaxis_title=x_label, yaxis_title='Accelerazione (unità non documentata)',
        legend_title_text='Segnale', hovermode='x unified', height=520)
    return fig


def lengths_figure(name):
    """Boxplot delle lunghezze (n_rows, in campioni) per classe, con i singoli file."""
    valid = V(name)
    classes = class_order(name)
    if valid.empty or not classes:
        return None

    palette = px.colors.qualitative.Safe
    fig = go.Figure()
    for index, class_name in enumerate(classes):
        group = valid[valid['class_name'] == class_name]
        fig.add_trace(go.Box(
            x=[class_name] * len(group), y=group['n_rows'],
            name=f'{class_name} (n={len(group)})',
            boxpoints='all', jitter=0.4, pointpos=0, boxmean=True,
            marker=dict(color=palette[index % len(palette)], size=4, opacity=0.65),
            customdata=group['file'],
            hovertemplate='file=%{customdata}<br>lunghezza=%{y} campioni<extra>' + class_name + '</extra>'))

    empty_classes = [c for c in cfg.DATASETS[name]['class_names'] if c not in classes]
    note = f' | classi senza registrazioni valide (omesse): {", ".join(empty_classes)}' if empty_classes else ''
    fig.update_layout(
        title=f'{name} | distribuzione delle lunghezze per classe<br><sup>{len(valid)} registrazioni valide{note}</sup>',
        xaxis_title='Classe', yaxis_title='Lunghezza (campioni)',
        showlegend=False, height=560,
        xaxis=dict(categoryorder='array', categoryarray=classes, tickangle=-35))
    return fig


class DatasetExplorer:
    """Widget con observer unici e un solo punto di visualizzazione del grafico."""

    def __init__(self):
        self._updating = False
        self._closed = False
        self._displayed = False
        layout = widgets.Layout(width='460px')
        datasets = [(f'{name} (n={len(V(name))})', name) for name in REC if len(V(name))]
        self.w_view = widgets.ToggleButtons(
            options=[('Registrazione', VIEW_RECORDING), ('Lunghezze per classe', VIEW_LENGTHS)],
            description='Vista:')
        self.w_ds = widgets.Dropdown(options=datasets, description='Dataset:', layout=layout)
        self.w_cls = widgets.Dropdown(description='Classe:', layout=layout)
        self.w_file = widgets.Dropdown(description='Registrazione:', layout=layout)
        self.w_random = widgets.Button(description='Registrazione casuale')
        self.w_max = widgets.BoundedIntText(
            value=0, min=0, max=10 ** 7, description='Max campioni:',
            style={'description_width': 'initial'})
        self.w_x = widgets.Dropdown(options=[X_INDEX], description='Asse x:', layout=layout)
        self.out = widgets.Output()
        self.recording_controls = widgets.VBox([
            widgets.HBox([self.w_cls]),
            widgets.HBox([self.w_file, self.w_random]),
            widgets.HBox([self.w_max, self.w_x]),
        ])
        self.box = widgets.VBox([self.w_view, self.w_ds, self.recording_controls, self.out])
        self._observers = [
            (self.w_ds, self._on_dataset), (self.w_cls, self._on_class),
            (self.w_file, self._on_file), (self.w_view, self._on_other),
            (self.w_max, self._on_other), (self.w_x, self._on_other)]
        self._widgets = [
            self.w_view, self.w_ds, self.w_cls, self.w_file, self.w_random,
            self.w_max, self.w_x, self.out, self.recording_controls, self.box]
        for widget, handler in self._observers:
            widget.observe(handler, names='value')
        self.w_random.on_click(self._on_random)
        self._refresh_classes()
        self._refresh_files()
        self._refresh_x()

    def close(self):
        """Rimuove callback e chiude tutti i widget prima di ricreare l'interfaccia."""
        if self._closed:
            return
        for widget, handler in self._observers:
            widget.unobserve(handler, names='value')
        self.w_random.on_click(self._on_random, remove=True)
        for widget in self._widgets:
            widget.close()
        self._closed = True

    def show(self):
        """Mostra una sola interfaccia anche se il metodo viene richiamato piu volte."""
        if self._closed:
            raise RuntimeError('L’esploratore è stato chiuso e non può essere riutilizzato.')
        if not self._displayed:
            display(self.box)
            self._displayed = True
        self._render()

    def _refresh_classes(self):
        self._updating = True
        try:
            if self.w_ds.value is None:
                self.w_cls.options = []
                return
            valid = V(self.w_ds.value)
            classes = class_order(self.w_ds.value)
            options = [(f'{class_name} (n={int((valid["class_name"] == class_name).sum())})', class_name)
                       for class_name in classes]
            previous = self.w_cls.value
            self.w_cls.options = options
            available = [value for _, value in options]
            self.w_cls.value = previous if previous in available else (available[0] if available else None)
        finally:
            self._updating = False

    def _refresh_files(self):
        self._updating = True
        try:
            if self.w_ds.value is None or self.w_cls.value is None:
                self.w_file.options = []
                return
            valid = V(self.w_ds.value)
            selected = valid[valid['class_name'] == self.w_cls.value]
            options = [(str(REC[self.w_ds.value].iloc[int(position)]['file']), int(position))
                       for position in selected.index]
            self.w_file.options = options
            self.w_file.value = options[0][1] if options else None
        finally:
            self._updating = False

    def _refresh_x(self):
        self._updating = True
        try:
            position = self.w_file.value
            has_extra = (position is not None and SIG[self.w_ds.value][position] is not None
                         and SIG[self.w_ds.value][position].shape[1] > AXIS_COLS)
            options = [X_INDEX, X_EXTRA] if has_extra else [X_INDEX]
            self.w_x.options = options
            if self.w_x.value not in options:
                self.w_x.value = X_INDEX
        finally:
            self._updating = False

    def _on_dataset(self, _):
        if self._updating:
            return
        self._refresh_classes()
        self._refresh_files()
        self._refresh_x()
        self._render()

    def _on_class(self, _):
        if self._updating:
            return
        self._refresh_files()
        self._refresh_x()
        self._render()

    def _on_file(self, _):
        if self._updating:
            return
        self._refresh_x()
        self._render()

    def _on_other(self, _):
        if not self._updating:
            self._render()

    def _on_random(self, _):
        options = [value for _, value in self.w_file.options]
        if options:
            self.w_file.value = random.choice(options)

    def _render(self):
        recording_view = self.w_view.value == VIEW_RECORDING
        self.recording_controls.layout.display = '' if recording_view else 'none'
        if self.w_ds.value is None:
            figure, message = None, 'Nessun dataset con registrazioni valide.'
        elif recording_view:
            figure = signal_figure(self.w_ds.value, self.w_file.value, self.w_max.value, self.w_x.value)
            message = 'Nessuna registrazione valida per la selezione.'
        else:
            figure = lengths_figure(self.w_ds.value)
            message = 'Nessuna lunghezza disponibile per il dataset selezionato.'

        # Unico punto di rendering: clear_output impedisce l'accumulo e display evita fig.show()/output impliciti.
        with self.out:
            clear_output(wait=True)
            if figure is None:
                print(message)
            else:
                display(figure)


if not (HAS_PLOTLY and HAS_WIDGETS) or not any(len(V(name)) for name in REC):
    print('Esploratore non disponibile: servono plotly, ipywidgets e almeno un dataset con registrazioni valide.')
else:
    if 'EXPLORER' in globals():
        EXPLORER.close()
    EXPLORER = DatasetExplorer()
    EXPLORER.show()


## 4. Controlli di qualità e struttura dei segnali

Controlli per registrazione sulle prime 3 colonne (assi), calcolati in memoria:

- valori non finiti, registrazioni vuote;
- canali **costanti** (massimo = minimo) e **quasi costanti** (deviazione standard < `NEAR_CONST_STD`, soglia **arbitraria** espressa nell'unità, non documentata, dei file);
- **righe duplicate** all'interno di una registrazione (stessi valori X, Y, Z su più campioni) e **registrazioni duplicate** (stesso contenuto in file diversi): sono due fenomeni distinti;
- sequenze di righe con X = Y = Z = 0 a inizio/fine: sono riportate solo come *osservazione*, **non** classificate come padding, perché possono essere valori reali del sensore.

In [10]:
NEAR_CONST_STD = 1e-3  # soglia arbitraria (unita dei file non documentata)


def qc_record(a):
    s = a[:, :AXIS_COLS]
    fin = np.isfinite(s)
    d = {'n_rows': len(s), 'n_nonfinite_axes': int((~fin).sum())}
    zero_rows = np.all(s == 0, axis=1)
    nz = ~zero_rows
    d['zero_rows'] = int(zero_rows.sum())
    d['lead_zero'] = int(np.argmax(nz)) if nz.any() else len(s)
    d['trail_zero'] = int(np.argmax(nz[::-1])) if nz.any() else len(s)
    d['dup_rows'] = int(len(s) - len(np.unique(s, axis=0)))
    n_const = n_near = 0
    for c in range(s.shape[1]):
        v = s[fin[:, c], c]
        d[f'min_{c}'] = float(v.min()) if v.size else np.nan
        d[f'max_{c}'] = float(v.max()) if v.size else np.nan
        if v.size:
            if v.max() == v.min():
                n_const += 1
            elif v.std() < NEAR_CONST_STD:
                n_near += 1
    d['n_const_ch'] = n_const
    d['n_nearconst_ch'] = n_near
    d['hash'] = hashlib.md5(np.ascontiguousarray(s).tobytes()).hexdigest()
    return d


QC = {}
for name in REC:
    rows = []
    for i, a in enumerate(SIG[name]):
        if a is None:
            continue
        d = qc_record(a)
        d.update(file=REC[name].at[i, 'file'], label=REC[name].at[i, 'label'], class_name=REC[name].at[i, 'class_name'])
        rows.append(d)
    QC[name] = pd.DataFrame(rows)


def qc_summary(name):
    q, rec = QC[name], REC[name]
    out = {
        'registrazioni analizzate': len(q),
        'vuote (0 righe)': int((rec['n_rows'] == 0).sum()),
        'errore di lettura': int(rec['read_error'].notna().sum()),
    }
    if q.empty:
        return out
    dup_groups = q.groupby('hash').filter(lambda g: len(g) > 1)
    n_lab = dup_groups.groupby('hash')['label'].nunique() if len(dup_groups) else pd.Series(dtype=int)
    out.update({
        'con valori non finiti': int((q['n_nonfinite_axes'] > 0).sum()),
        'con >=1 canale costante': int((q['n_const_ch'] > 0).sum()),
        'con >=1 canale quasi costante': int((q['n_nearconst_ch'] > 0).sum()),
        'con righe duplicate interne': int((q['dup_rows'] > 0).sum()),
        'quota mediana righe duplicate (%)': round(float((100 * q['dup_rows'] / q['n_rows']).median()), 1),
        'con righe iniziali tutte nulle (X=Y=Z=0)': int((q['lead_zero'] > 0).sum()),
        'con righe finali tutte nulle (X=Y=Z=0)': int((q['trail_zero'] > 0).sum()),
        'interamente nulle': int((q['lead_zero'] == q['n_rows']).sum()),
        'gruppi di registrazioni duplicate': int(dup_groups['hash'].nunique()) if len(dup_groups) else 0,
        'registrazioni in gruppi duplicati': len(dup_groups),
        'gruppi duplicati con label diverse': int((n_lab > 1).sum()),
    })
    return out


QC_SUM = {n: qc_summary(n) for n in REC}
print('Riepilogo controlli di qualita (conteggi di registrazioni)')
display(pd.DataFrame(QC_SUM))

ch_rows = []
for name in REC:
    q = QC[name]
    if q.empty:
        continue
    n_ch = int(REC[name]['n_cols'].mode().iloc[0])
    names = channel_names(max(n_ch, AXIS_COLS))
    for c in range(AXIS_COLS):
        if f'min_{c}' in q:
            ch_rows.append({'dataset': name, 'canale': names[c], 'minimo globale': q[f'min_{c}'].min(),
                            'massimo globale': q[f'max_{c}'].max(),
                            'mediana dei minimi per registrazione': q[f'min_{c}'].median(),
                            'mediana dei massimi per registrazione': q[f'max_{c}'].median()})
print('\nValori minimi e massimi dei canali')
display(pd.DataFrame(ch_rows, columns=['dataset', 'canale', 'minimo globale', 'massimo globale',
                             'mediana dei minimi per registrazione', 'mediana dei massimi per registrazione'])
        .set_index(['dataset', 'canale']))

for name in REC:
    q = QC[name]
    if q.empty:
        continue
    for col, txt in (('trail_zero', 'righe finali nulle'), ('lead_zero', 'righe iniziali nulle')):
        top = q[q[col] > 0].sort_values(col, ascending=False).head(5)
        if len(top):
            print(f'\n{name}: prime 5 registrazioni per {txt} (osservazione, NON necessariamente padding)')
            display(top[['file', 'class_name', 'n_rows', 'lead_zero', 'trail_zero']])
        else:
            print(f'\n{name}: nessuna registrazione con {txt}.')

Riepilogo controlli di qualita (conteggi di registrazioni)


,UniZgFall,UniMiB
registrazioni analizzate,468.0,1980.0
vuote (0 righe),0.0,0.0
errore di lettura,0.0,0.0
con valori non finiti,0.0,0.0
con >=1 canale costante,0.0,0.0
con >=1 canale quasi costante,0.0,0.0
con righe duplicate interne,461.0,65.0
quota mediana righe duplicate (%),2.0,0.0
con righe iniziali tutte nulle (X=Y=Z=0),0.0,0.0
con righe finali tutte nulle (X=Y=Z=0),0.0,0.0



Valori minimi e massimi dei canali


minimo globale  massimo globale  mediana dei minimi per registrazione  mediana dei massimi per registrazione
dataset   canale                                                                                                              
UniZgFall X            -8.097400         7.820800                             -1.079900                               1.080350
          Y            -7.967900         8.034000                             -0.663945                               2.688850
          Z            -7.527700         8.161800                             -0.457970                               1.094900
UniMiB    X           -19.581081        19.543959                            -11.613150                              11.020482
          Y           -19.581081        19.543959                            -12.914969                              11.551889
          Z           -19.543959        19.581081                            -12.626818                              11.236940


UniZgFall: nessuna registrazione con righe finali nulle.

UniZgFall: nessuna registrazione con righe iniziali nulle.

UniMiB: nessuna registrazione con righe finali nulle.

UniMiB: nessuna registrazione con righe iniziali nulle.


### Registrazioni rappresentative (lunghezza minima, mediana, massima)

Per ciascun dataset, prima le tre registrazioni più vicine alla lunghezza minima, mediana e massima **dell'intero dataset**.

In [11]:
def representatives(name):
    """Registrazioni più vicine alla lunghezza minima, mediana e massima del dataset."""
    v = V(name)
    if v.empty:
        return []
    return [(tag, int((v['n_rows'] - target).abs().idxmin()))
            for tag, target in (('minima', v['n_rows'].min()), ('mediana', v['n_rows'].median()), ('massima', v['n_rows'].max()))]


def representative_figure(name):
    picks = representatives(name)
    if not picks:
        return None
    r = REC[name]
    titles = [f'lunghezza {tag}: {r.at[i, "file"]} ({int(r.at[i, "n_rows"])} campioni, classe {r.at[i, "class_name"]})' for tag, i in picks]
    fig = make_subplots(rows=len(picks), cols=1, subplot_titles=titles, vertical_spacing=0.1)
    for k, (tag, i) in enumerate(picks, start=1):
        a = SIG[name][i]
        names = channel_names(a.shape[1])
        for c in range(min(AXIS_COLS, a.shape[1])):
            fig.add_trace(go.Scatter(
                x=np.arange(len(a)), y=a[:, c], mode='lines', name=names[c], legendgroup=names[c],
                showlegend=(k == 1), line=dict(color=AXIS_COLORS[c], width=1),
                hovertemplate='campione=%{x}<br>' + names[c] + '=%{y:.5f}<extra></extra>'), row=k, col=1)
        fig.update_xaxes(title_text='indice del campione', row=k, col=1)
    fig.update_layout(title=f'{name} | registrazioni rappresentative (tutto il dataset) | unità non documentate',
                      height=280 * len(picks) + 120, legend_title_text='Canale')
    return fig


if not HAS_PLOTLY:
    print('Plotly non disponibile: grafici omessi.')
else:
    for name in REC:
        fig = representative_figure(name)
        if fig is not None:
            fig.show()


In [12]:
print('Padding gia applicato ai file? Recuperabilita della lunghezza originale')
for name in REC:
    print('=' * 88)
    print(name)
    q, dv = QC[name], DERIVED.get(name, {})
    print(f'  CSV originali (osservato): lunghezze variabili ({int(LEN_ALL.loc[name, "n_lunghezze_distinte"])} valori distinti) -> non risultano uniformati con padding.')
    if not q.empty:
        n_tz = int((q['trail_zero'] > 0).sum())
        print(f'  righe finali X=Y=Z=0 nei CSV originali (osservato): {n_tz} registrazioni su {len(q)}; poiche i file originali non sono padded, queste righe nulle sono dati dei file, non padding.')
    if dv.get('stato') == 'ok':
        print(f'  array derivato (osservato): la lunghezza originale e recuperabile da raw_{name}_meta.csv (n_samples coincidente in {dv["n_samples == lunghezza CSV originale"]}/{dv["registrazioni confrontate"]} casi).')
        if not q.empty and (q['trail_zero'] > 0).any():
            print('  LIMITE: per le registrazioni con righe finali nulle reali, dal solo array raw_*_X.npy (senza meta) non si puo distinguere zero reale da padding.')
        else:
            print('  LIMITE (ipotesi): senza meta, uno zero finale non e comunque garantito essere padding, perche i valori nulli possono essere reali.')
    else:
        print('  confronto con i file derivati non eseguito:', dv.get('stato'), '-> recuperabilita non verificata.')

Padding gia applicato ai file? Recuperabilita della lunghezza originale
UniZgFall
  CSV originali (osservato): lunghezze variabili (427 valori distinti) -> non risultano uniformati con padding.
  righe finali X=Y=Z=0 nei CSV originali (osservato): 0 registrazioni su 468; poiche i file originali non sono padded, queste righe nulle sono dati dei file, non padding.
  array derivato (osservato): la lunghezza originale e recuperabile da raw_UniZgFall_meta.csv (n_samples coincidente in 468/468 casi).
  LIMITE (ipotesi): senza meta, uno zero finale non e comunque garantito essere padding, perche i valori nulli possono essere reali.
UniMiB
  CSV originali (osservato): lunghezze variabili (558 valori distinti) -> non risultano uniformati con padding.
  righe finali X=Y=Z=0 nei CSV originali (osservato): 0 registrazioni su 1980; poiche i file originali non sono padded, queste righe nulle sono dati dei file, non padding.
  array derivato (osservato): la lunghezza originale e recuperabile da raw_U

## 5. Frequenza di campionamento e unità di misura

La cella cerca nella documentazione già presente (`README.md` del progetto, `README.md` e `ANALISI_TECNICA.md` della cartella sorgente, in sola lettura) le righe che citano frequenza o unità, analizza la 4ª colonna di UniZgFall e confronta l'ordine di grandezza del modulo del vettore. **Non** si stima la frequenza di campionamento dal numero di campioni.

In [13]:
DOC_FILES = [cfg.PROJECT_DIR / 'README.md', cfg.SOURCE_DIR / 'README.md', cfg.SOURCE_DIR / 'ANALISI_TECNICA.md']
KEYWORDS = ['sample rate', 'sampling', 'campionament', 'frequenz', ' hz', 'unità', 'm/s']
hits = []
for p in DOC_FILES:
    if not p.is_file():
        hits.append({'file': rel(p), 'riga': None, 'testo': 'file non trovato'})
        continue
    for ln, line in enumerate(p.read_text(encoding='utf-8', errors='replace').splitlines(), start=1):
        low = line.lower()
        if any(k in low for k in KEYWORDS):
            hits.append({'file': rel(p), 'riga': ln, 'testo': line.strip()[:220]})
HITS = pd.DataFrame(hits, columns=['file', 'riga', 'testo'])
NEG = ('non document', 'non specific', 'non è documentat', 'non sono documentat', 'non disponibil', 'non verificabil')
n_neg = int(HITS['testo'].str.lower().map(lambda t: any(k in t for k in NEG)).sum()) if len(HITS) else 0
print(f'Righe di documentazione che citano frequenza/unita: {len(HITS)} (di cui {n_neg} con diciture di assenza/non verificabilita). Prime 25:')
display(HITS.head(25))

TIMECOL = {}
for name in REC:
    steps, mono, n_ext = [], 0, 0
    for a in SIG[name]:
        if a is None or a.shape[1] <= AXIS_COLS:
            continue
        n_ext += 1
        dt = np.diff(a[:, AXIS_COLS])
        if dt.size and np.all(np.isfinite(dt)):
            mono += int(np.all(dt > 0))
            steps.append(float(np.median(dt)))
    TIMECOL[name] = {'n_con_colonna_extra': n_ext, 'n_strettamente_crescenti': mono,
                     'passo_mediano_min': min(steps) if steps else np.nan,
                     'passo_mediano_mediana': float(np.median(steps)) if steps else np.nan,
                     'passo_mediano_max': max(steps) if steps else np.nan}
print('\nColonna extra (4a) dei file: passo tra righe consecutive (valori del file, unita non documentata)')
display(pd.DataFrame(TIMECOL).T)

MAGN = {}
for name in REC:
    meds = []
    for a in SIG[name]:
        if a is None or a.shape[1] < AXIS_COLS:
            continue
        values = np.linalg.norm(a[:, :AXIS_COLS], axis=1)
        finite = values[np.isfinite(values)]
        if finite.size:
            meds.append(float(np.median(finite)))
    MAGN[name] = float(np.median(meds)) if meds else np.nan
print('\nMediana (sulle registrazioni) della mediana del modulo |(X,Y,Z)|:', {k: round(v, 3) for k, v in MAGN.items()})

rows = []
for name in REC:
    rec, tc = REC[name], TIMECOL[name]
    n_hdr = int(rec['header'].fillna('').str.startswith('#').sum())
    hdr_txt = rec['header'].fillna('')[rec['header'].fillna('').str.startswith('#')].unique().tolist()
    if n_hdr:
        rows.append({'stato': 'VERIFICATA', 'dataset': name, 'informazione': f'intestazione dei file: {hdr_txt} ({n_hdr} file)', 'evidenza': 'prima riga dei CSV letta direttamente'})
    else:
        rows.append({'stato': 'VERIFICATA', 'dataset': name, 'informazione': 'nessuna intestazione nei file', 'evidenza': 'prima riga dei CSV letta direttamente'})
    if tc['n_con_colonna_extra']:
        rows.append({'stato': 'VERIFICATA', 'dataset': name,
                     'informazione': f'{tc["n_con_colonna_extra"]} file hanno una 4a colonna; strettamente crescente in {tc["n_strettamente_crescenti"]}; passo mediano per file da {tc["passo_mediano_min"]:.4g} a {tc["passo_mediano_max"]:.4g}',
                     'evidenza': 'calcolo su np.diff della 4a colonna'})
        rows.append({'stato': 'DEDOTTA', 'dataset': name,
                     'informazione': 'la 4a colonna e un asse temporale',
                     'evidenza': 'monotonia osservata + assunzione del README del progetto; unita (s? ms?) e frequenza NON verificate'})
    else:
        rows.append({'stato': 'NON DISPONIBILE', 'dataset': name, 'informazione': 'asse temporale/timestamp nei file', 'evidenza': 'i file hanno solo le colonne degli assi'})
    m = MAGN[name]
    if np.isfinite(m):
        if 8.5 <= m <= 11:
            guess = 'compatibile con m/s^2 (gravita ~9.8)'
        elif 0.8 <= m <= 1.3:
            guess = 'compatibile con g (gravita ~1)'
        else:
            guess = 'ordine di grandezza non riconducibile con chiarezza a g o m/s^2'
        rows.append({'stato': 'DEDOTTA', 'dataset': name, 'informazione': f'unita: {guess}',
                     'evidenza': f'modulo mediano del vettore = {m:.3f}; ipotesi debole: dipende da orientamento, tipo di sensore, eventuale rimozione della gravita'})
    rows.append({'stato': 'NON DISPONIBILE', 'dataset': name, 'informazione': 'frequenza di campionamento', 'evidenza': 'non presente nei file ne (nelle righe trovate) nella documentazione locale'})
    rows.append({'stato': 'NON DISPONIBILE', 'dataset': name, 'informazione': 'unita di misura certificata, calibrazione, orientamento e posizione del sensore', 'evidenza': 'non presenti nei file ne nella documentazione locale'})
INFO = pd.DataFrame(rows, columns=['stato', 'dataset', 'informazione', 'evidenza'])
print('\nSintesi: informazioni verificate / dedotte / non disponibili')
display(INFO.sort_values(['stato', 'dataset']).reset_index(drop=True))

print('\nNumero di bin FFT usati dalle feature (utils.extract_features: len//2 - 1) alle lunghezze osservate')
fft_rows = {n: {'bin con lunghezza minima': int(LEN_ALL.loc[n, 'min']) // 2 - 1, 'bin con lunghezza mediana': int(LEN_ALL.loc[n, 'mediana']) // 2 - 1, 'bin con lunghezza massima': int(LEN_ALL.loc[n, 'max']) // 2 - 1} for n in LEN_ALL.index}
display(pd.DataFrame(fft_rows).T)

Righe di documentazione che citano frequenza/unita: 11 (di cui 3 con diciture di assenza/non verificabilita). Prime 25:


,file,riga,testo
0,README.md,38,- Frequenza di campionamento e unità di misura (g o m/s²) di entrambi i dataset: non documentate nei file.
1,README.md,86,"Raw input: i file sono ricostruiti dall'ordine di flatten preesistente in `(N, 3, L)`, mantenendo le colonne x/y/z e il padding a destra già generato. MiniRocket/MultiRocket richiedono lunghezze u..."
2,../ml_accelerometers/ANALISI_TECNICA.md,56,"Le registrazioni contengono da 388 a 5.146 campioni (mediana 1.525). Il campo temporale di alcuni record avanza di circa 0,005 tra righe, ma il repository non documenta unità, frequenza nominale, ..."
3,../ml_accelerometers/ANALISI_TECNICA.md,77,"`UniMiB_raw` contiene 1.980 CSV con nomi `subject_0_file_0.csv` ... `subject_29_file_65.csv`: si verificano 30 identificativi di soggetto, ciascuno con 66 file. I CSV hanno tre colonne `x,y,z`; l'..."
4,../ml_accelerometers/ANALISI_TECNICA.md,134,Per ogni asse viene poi calcolato `abs(np.fft.fft(s))[1:n//2]`: si scartano il bin DC e la metà superiore dello spettro. La trasformata non è normalizzata e non è convertita in frequenza fisica pe...
5,../ml_accelerometers/ANALISI_TECNICA.md,136,"Il totale è dunque \(48+2+42+2=94\) scalari per record. In termini di nomi, ogni statistica per asse è prefissata da `x_`, `y_` o `z_`; le versioni spettrali sono suffissate da `_fft`. Non vengono..."
6,../ml_accelerometers/ANALISI_TECNICA.md,289,"La comparazione features-vs-raw all'interno di un dataset utilizza, in linea generale, lo stesso vettore `y` e la stessa configurazione StratifiedKFold, ma diversa dimensionalità e preprocessing, ..."
7,../ml_accelerometers/ANALISI_TECNICA.md,306,"| Non sono documentati sample rate, unità, orientamento, protocollo di acquisizione o ambiente reale per entrambi i dataset | Trasferibilità al dispositivo e comportamento online non sono dimostra..."
8,../ml_accelerometers/ANALISI_TECNICA.md,307,"| Il helper `CI95` e i p-value sono salvati, ma il nome “CI95” non corrisponde a un CI convenzionale per la media | Rischio di sovrainterpretare la precisione degli score | Media | Confermato dal ..."
9,../ml_accelerometers/ANALISI_TECNICA.md,338,"7. **Documentare dati e feature**: legenda label UniZgFall, espansione delle sigle UniMiB, frequenza/unità, modalità e soggetti; valutare stabilità delle feature per soggetto e importanza permutat..."



Colonna extra (4a) dei file: passo tra righe consecutive (valori del file, unita non documentata)


,n_con_colonna_extra,n_strettamente_crescenti,passo_mediano_min,passo_mediano_mediana,passo_mediano_max
UniZgFall,468.0,367.0,0.0,0.005,0.005
UniMiB,0.0,0.0,NaN,NaN,NaN



Mediana (sulle registrazioni) della mediana del modulo |(X,Y,Z)|: {'UniZgFall': 1.005, 'UniMiB': 9.294}

Sintesi: informazioni verificate / dedotte / non disponibili


,stato,dataset,informazione,evidenza
0,DEDOTTA,UniMiB,unita: compatibile con m/s^2 (gravita ~9.8),"modulo mediano del vettore = 9.294; ipotesi debole: dipende da orientamento, tipo di sensore, eventuale rimozione della gravita"
1,DEDOTTA,UniZgFall,la 4a colonna e un asse temporale,monotonia osservata + assunzione del README del progetto; unita (s? ms?) e frequenza NON verificate
2,DEDOTTA,UniZgFall,unita: compatibile con g (gravita ~1),"modulo mediano del vettore = 1.005; ipotesi debole: dipende da orientamento, tipo di sensore, eventuale rimozione della gravita"
3,NON DISPONIBILE,UniMiB,asse temporale/timestamp nei file,i file hanno solo le colonne degli assi
4,NON DISPONIBILE,UniMiB,frequenza di campionamento,non presente nei file ne (nelle righe trovate) nella documentazione locale
5,NON DISPONIBILE,UniMiB,"unita di misura certificata, calibrazione, orientamento e posizione del sensore",non presenti nei file ne nella documentazione locale
6,NON DISPONIBILE,UniZgFall,frequenza di campionamento,non presente nei file ne (nelle righe trovate) nella documentazione locale
7,NON DISPONIBILE,UniZgFall,"unita di misura certificata, calibrazione, orientamento e posizione del sensore",non presenti nei file ne nella documentazione locale
8,VERIFICATA,UniMiB,"intestazione dei file: ['# x,y,z'] (1980 file)",prima riga dei CSV letta direttamente
9,VERIFICATA,UniZgFall,nessuna intestazione nei file,prima riga dei CSV letta direttamente



Numero di bin FFT usati dalle feature (utils.extract_features: len//2 - 1) alle lunghezze osservate


,bin con lunghezza minima,bin con lunghezza mediana,bin con lunghezza massima
UniZgFall,193,761,2572
UniMiB,41,102,930


### Effetto delle differenze di lunghezza su feature temporali e FFT

*Verificato leggendo `utils.py` (`_stats`, `extract_features`):*

- `neg_count`, `pos_count`, `above_mean` e `peak_count` sono **conteggi non normalizzati**: crescono con il numero di campioni.
- Le FFT sono calcolate con `np.abs(np.fft.fft(...))` **senza normalizzazione** e usando i bin `1 .. n//2 - 1`: il numero di bin e l'ampiezza dello spettro dipendono dalla lunghezza (si veda la tabella sopra), quindi minimo, massimo e somme dei moduli non sono direttamente confrontabili tra registrazioni di lunghezza diversa.
- Senza frequenza di campionamento, i bin FFT non possono essere convertiti in Hz e la risoluzione in frequenza (che dipende da lunghezza e frequenza) non è confrontabile.
- Media, deviazione standard, mediana, `energy` (divisa per *n*) non dipendono direttamente da *n*, ma dipendono dal contenuto della finestra (una caduta breve in una finestra lunga è «diluita»).

*Verifiche necessarie prima di confrontare i segnali (ipotesi di lavoro, non eseguite qui):* frequenza di campionamento e unità di ciascun dataset; se le registrazioni sono segmentate attorno all'evento o contengono fasi di riposo; relazione tra lunghezza e classe (vedi Sezione 2); stabilità delle feature al variare della lunghezza (ad esempio ripetendo il calcolo su finestre di lunghezza fissa).

## 6. Riepilogo delle osservazioni

I punti seguenti sono calcolati dai dati caricati e distinguono risultati osservati da ipotesi; non vengono anticipati valori o conclusioni statiche.

Sintesi generata dai valori effettivamente calcolati nelle sezioni precedenti. Ogni riga è marcata `[OSSERVATO]` (calcolato) oppure `[IPOTESI]` (da verificare).

In [14]:
def line(tag, text):
    print(f'  [{tag}] {text}')


names_ok = [n for n in LEN_ALL.index]
uniform = [n for n in names_ok if int(LEN_ALL.loc[n, 'n_lunghezze_distinte']) == 1]
variable = [n for n in names_ok if n not in uniform]

print('1) Dataset con registrazioni di lunghezza uniforme')
line('OSSERVATO', ', '.join(uniform) if uniform else 'nessuno')

print('2) Dataset con serie di lunghezza variabile')
for n in variable:
    r = LEN_ALL.loc[n]
    line('OSSERVATO', f'{n}: {int(r["n_lunghezze_distinte"])} lunghezze distinte, da {int(r["min"])} a {int(r["max"])} (mediana {r["mediana"]:.0f})')

print('3) Classi con la maggiore variabilita di lunghezza (coefficiente di variazione, prime 3)')
for n, cs in CLASS_STATS.items():
    top = cs['cv'].dropna().sort_values(ascending=False).head(3)
    txt = '; '.join(f'{idx[1]} (CV={val:.2f}, min-max {int(cs.loc[idx, "min"])}-{int(cs.loc[idx, "max"])})' for idx, val in top.items())
    line('OSSERVATO', f'{n}: {txt}')

print('4) Padding gia applicato nei file?')
for n in REC:
    dv = DERIVED.get(n, {})
    line('OSSERVATO', f'{n}: CSV originali con lunghezza variabile -> nessuna uniformazione rilevata nei file originali.')
    if dv.get('stato') == 'ok':
        line('OSSERVATO', f'{n}: il padding esiste negli array derivati data/raw_{n}_X.npy (fino a {dv["lunghezza_con_padding"]} campioni, ~{dv["quota di campioni di padding (%)"]}% di padding); lunghezza originale recuperabile da n_samples in {dv["n_samples == lunghezza CSV originale"]}/{dv["registrazioni confrontate"]} casi.')
    else:
        line('IPOTESI', f'{n}: file derivati non verificati ({dv.get("stato")}).')

print('5) Anomalie effettivamente rilevate (conteggi diversi da zero)')
any_anom = False
for n in REC:
    items = {k: v for k, v in {**anom[n], **QC_SUM[n]}.items()
             if isinstance(v, (int, np.integer)) and v > 0 and k not in ('file totali', 'registrazioni analizzate')}
    if items:
        any_anom = True
        line('OSSERVATO', f'{n}: ' + '; '.join(f'{k} = {v}' for k, v in items.items()))
    else:
        line('OSSERVATO', f'{n}: nessuna anomalia tra i controlli implementati')
if any_anom:
    line('IPOTESI', 'righe duplicate o nulle e canali (quasi) costanti possono essere reali (sensore fermo, saturazione); non vengono considerate errori senza ulteriori prove.')

print('6) Informazioni mancanti per interpretare correttamente i segnali')
missing_info = INFO[INFO['stato'] == 'NON DISPONIBILE'].groupby('informazione')['dataset'].apply(lambda s: ', '.join(sorted(set(s))))
for k, v in missing_info.items():
    line('OSSERVATO', f'{k} ({v})')
line('OSSERVATO', 'UniZgFall: identificativo del soggetto non disponibile; significato delle sigle di classe non documentato nel repository.')

print('7) Aspetti da verificare prima di confrontare modelli sui segnali grezzi')
for n, e in ETA2.items():
    line('OSSERVATO', f'{n}: la classe spiega una frazione eta^2 = {e:.2f} della varianza della lunghezza.')
line('IPOTESI', 'se la lunghezza e legata alla classe, la quantita di padding (zeri in coda) puo diventare un indizio involontario per modelli che non la mascherano: da verificare con confronti a parita di trattamento del padding.')
line('IPOTESI', 'modelli diversi trattano il padding in modo diverso (nessuna maschera, maschera solo nel pooling, ecc.): verificare caso per caso prima di confrontare i risultati.')
line('IPOTESI', 'frequenza di campionamento e unita non note: lunghezze e feature FFT non sono confrontabili tra UniMiB e UniZgFall senza questa informazione.')
line('IPOTESI', 'UniMiB ha identificativi di soggetto, UniZgFall no: la separazione per soggetto nella validazione e verificabile solo per UniMiB.')

1) Dataset con registrazioni di lunghezza uniforme
  [OSSERVATO] nessuno
2) Dataset con serie di lunghezza variabile
  [OSSERVATO] UniZgFall: 427 lunghezze distinte, da 388 a 5146 (mediana 1525)
  [OSSERVATO] UniMiB: 558 lunghezze distinte, da 85 a 1863 (mediana 206)
3) Classi con la maggiore variabilita di lunghezza (coefficiente di variazione, prime 3)
  [OSSERVATO] UniZgFall: FF (CV=0.29, min-max 840-2707); OT (CV=0.29, min-max 388-2112); MJ (CV=0.24, min-max 593-1891)
  [OSSERVATO] UniMiB: StandingUpFL (CV=0.43, min-max 425-1746); LyingDownFS (CV=0.40, min-max 382-1678); Jumping (CV=0.34, min-max 85-1159)
4) Padding gia applicato nei file?
  [OSSERVATO] UniZgFall: CSV originali con lunghezza variabile -> nessuna uniformazione rilevata nei file originali.
  [OSSERVATO] UniZgFall: il padding esiste negli array derivati data/raw_UniZgFall_X.npy (fino a 5150 campioni, ~67.0% di padding); lunghezza originale recuperabile da n_samples in 468/468 casi.
  [OSSERVATO] UniMiB: CSV originali 